# KKB Agentic Data Analytics: güncel veri doğrulama notebook'u

**8 Eylül 2026 veri kapanış sürümü**

Bu notebook eski 25 serilik başlangıç snapshot'ı değildir. Repodaki güncel,
self-contained DuckDB dosyasını salt okunur açar ve BDDK aylık, BDDK haftalık,
BDDK FinTürk, seçilmiş ulusal ve bölgesel TCMB EVDS serileri, TÜİK il konut
satışları, TBB tüketici kredileri, hanehalkı finansmanı ve resmî bağlam
belgelerinin birleşik durumunu gösterir.

Ana sonuç: Zorunlu kaynak ailelerinin yayımlanmış veri kapsamı tamamlandı.
Kaynakta henüz yayımlanmayan TBB Haziran 2026 raporu tahmin edilmedi ve açık
boşluk olarak korunuyor.


## Kapsam

| Kaynak | Yerel kapsam |
| --- | --- |
| BDDK aylık | 66 ay, 10 resmî banka grubu, 17 tablonun tamamı, 1.334.850 semantik ölçüm |
| BDDK haftalık | 286 hafta, 7 resmî banka grubu, 9 tablonun tamamı, 1.025.974 ölçüm |
| BDDK FinTürk | 22 çeyrek, 7 tablo, 7 banka grubu, 81 il ve `YURT DIŞI`, 936.512 ölçüm, 1.328 kaynak-null şube değeri için auditli analitik sıfır |
| TCMB EVDS | 52.696 serilik metadata kataloğu, analitik değeri yüksek 506 kaynak serinin yerel gözlemi ve 1 türetilmiş altın serisi |
| TÜİK il konut satışları | 81 il, Ocak 2020-Haziran 2026, 5 aylık satış metriği ve EVDS çapraz doğrulaması |
| İl bazlı konut paneli | 81 il, 22 çeyrek, 1782 tekil satır ve 1620 analize hazır satır |
| TBB | Mart 2021-Mart 2026 arasında yayımlanmış 21 rapor, 252 ürün ölçümü |
| Resmî belgeler | 4 BDDK kararı ve 4 TCMB yöntem veya destek belgesi |

EVDS katalog kaydı, gözlemin indirildiği anlamına gelmez. Bu ayrım
`observation_available` alanında açıkça tutulur.


In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    path for path in candidates
    if (path / "data_pipeline" / "lakehouse" / "analytics.duckdb").exists()
)
DB_PATH = ROOT / "data_pipeline" / "lakehouse" / "analytics.duckdb"
connection = duckdb.connect(str(DB_PATH), read_only=True)
print(f"DuckDB: {DB_PATH}")
print(f"Boyut: {DB_PATH.stat().st_size / 1024 / 1024:.2f} MiB")


DuckDB: data_pipeline/lakehouse/analytics.duckdb
Boyut: 98.26 MiB


## Birleşik katalog özeti

In [ ]:
connection.execute(
    """
    SELECT
      (SELECT count(*) FROM catalog.data_assets) AS veri_varligi,
      (SELECT count(*) FROM catalog.metrics) AS katalog_metrigi,
      (SELECT count(*) FROM catalog.metrics
       WHERE observation_available) AS sorgulanabilir_metrik,
      (SELECT count(*) FROM evds.series_catalog) AS evds_katalog_serisi,
      (SELECT count(*) FROM catalog.metrics
       WHERE source_system = 'TCMB_EVDS'
         AND observation_available) AS yerel_evds_serisi,
      (SELECT count(*) FROM analysis.housing_credit_monthly) AS aylik_donem,
      (SELECT count(*) FROM analysis.housing_credit_quarterly) AS ceyreklik_donem,
      (SELECT count(*) FROM regional.housing_quarterly) AS bolgesel_satir,
      (SELECT count(DISTINCT province_key)
       FROM regional.housing_quarterly) AS il_sayisi,
      (SELECT count(*) FROM regional.housing_quarterly
       WHERE analysis_ready) AS analize_hazir_bolgesel_satir
    """
).fetchdf()


veri_varligi,katalog_metrigi,sorgulanabilir_metrik,evds_katalog_serisi,yerel_evds_serisi,aylik_donem,ceyreklik_donem,bolgesel_satir,il_sayisi,analize_hazir_bolgesel_satir
66,55489,3297,52696,506,66,22,1782,81,1620


## Kaynak varlıklarının yükleme durumu

In [ ]:
connection.execute(
    """
    SELECT source_system AS kaynak,
           string_agg(DISTINCT status, ', ' ORDER BY status) AS durumlar,
           count(*) AS varlik,
           CAST(sum(row_count) AS BIGINT) AS kayit
    FROM catalog.data_assets
    GROUP BY source_system
    ORDER BY source_system
    """
).fetchdf()


kaynak,durumlar,varlik,kayit
BDDK_FINTURK,passed,10,1022371
BDDK_MONTHLY,passed,18,1674500
BDDK_WEEKLY,"complete, passed",3,1043994
CROSS_SOURCE_QUALITY,passed_with_expected_scope_differences,1,22
OFFICIAL_DOCUMENTS,passed,1,5
REGIONAL_HOUSING_ANALYSIS,passed_with_source_gaps,3,1889
TBB_REPORTS,passed_with_source_gaps,5,44326
TCMB_EVDS,"complete, passed",21,96197
TUIK_DATA_PORTAL,passed,4,326148


## Analitik sözleşme

- Kredi stoku, stok değişimi ve yeni kullandırım akımı farklı ölçülerdir.
- Haftalık faiz aylığa çevrilirken kullanılan yöntem metadata ile birlikte saklanır.
- Kümülatif BDDK kâr-zarar değerleri kaynak hâliyle korunur, türetilmiş aylık akım ayrıca tutulur.
- Eksik değerler tahminle doldurulmaz. Sıfır yalnız aynı resmî kaynak içindeki
  kesin bir toplamsal kimlikle kanıtlanırsa, ham null korunarak ayrı provenance
  ile kullanılabilir.
- FinTürk'teki 1.328 kaynak-null şube değeri, 1.782 il-çeyreğin tamamında sıfır
  farkla geçen `SEKTÖR = MEVDUAT + KATILIM + KALKINMA VE YATIRIM` kimliğiyle
  ayrı `usable_value=0` ve audit kaydı olarak tutulur.
- BDDK haftalık kaynak boşlukları ile FinTürk yapısal boşlukları ayrı denetim
  tablolarında tutulur.
- Çeyreklik gözlem ara aylara forward fill edilmez.
- İl bazlı panelde eksik aylı çeyrekler kısmi toplamla doldurulmaz.
- İpoteksiz satış, nakit satış olarak yorumlanmaz.
- Birlikte hareket nedensellik kanıtı sayılmaz. Olay belgeleri yalnız araştırma bağlamıdır.


## Güncel aylık analiz tablosundan örnek

In [ ]:
connection.execute(
    """
    SELECT month AS ay,
           bddk_housing_credit_stock_million_tl AS bddk_konut_kredisi_milyon_tl,
           round(TP_KTF12, 2) AS konut_kredisi_faizi_yuzde,
           round(TP_TUKFIY2025_GENEL, 2) AS tufe_endeksi,
           round(TP_KFE_TR, 2) AS konut_fiyat_endeksi,
           round(mortgaged_sales_share_pct, 2) AS ipotekli_satis_payi_yuzde
    FROM analysis.housing_credit_monthly
    ORDER BY month DESC
    LIMIT 12
    """
).fetchdf()


ay,bddk_konut_kredisi_milyon_tl,konut_kredisi_faizi_yuzde,tufe_endeksi,konut_fiyat_endeksi,ipotekli_satis_payi_yuzde
2026-06,801438,40.34,129.99,231.34,20.00
2026-05,782918,38.43,128.72,226.99,21.17
2026-04,767314,36.62,126.55,223.32,20.32
2026-03,741972,34.66,121.47,219.52,22.91
2026-02,715799,35.16,119.16,215.40,20.10
2026-01,691343,36.24,115.73,211.73,18.18
2025-12,678970,37.29,110.39,204.36,11.54
2025-11,658770,37.76,109.42,204.04,15.32
2025-10,642144,37.94,108.48,198.87,14.44
2025-09,623765,38.80,105.78,195.66,14.19


## Demo koşulu: faiz düşerken reel kredi stokunun artmadığı aylar

Bu filtre betimleyici bir eş hareket bulgusudur, tek başına nedensellik kanıtı değildir.

In [ ]:
demo = connection.execute(
    """
    SELECT month AS ay,
           round(housing_rate_change_pp, 3) AS faiz_degisimi_puan,
           round(housing_credit_real_mom_pct, 3) AS reel_kredi_aylik_yuzde,
           source_scope_review_required AS kaynak_kapsami_incele
    FROM analysis.housing_credit_monthly
    WHERE rate_down_real_stock_not_up_quality_screened
    ORDER BY month
    """
).fetchdf()
print(f"Koşulu sağlayan kalite taramalı ay sayısı: {len(demo)}")
display(demo)


ay,faiz_degisimi_puan,reel_kredi_aylik_yuzde,kaynak_kapsami_incele
2021-02,-0.403,-1.087,False
2021-03,-0.250,-0.912,False
2021-05,-0.299,-1.310,False
2021-08,-0.072,-0.335,False
2021-10,-0.075,-1.456,False
2021-11,-0.723,-1.275,False
2022-02,-0.448,-3.535,False
2022-06,-0.242,-0.355,False
2022-12,-1.011,-0.109,False
2023-02,-0.637,-1.660,False


## Kaynak kapsamlarının çeyreklik karşılaştırması

In [ ]:
connection.execute(
    """
    SELECT quarter AS ceyrek,
           bddk_housing_credit_stock_million_tl AS bddk_milyon_tl,
           round(finturk_domestic_minus_bddk_pct, 4) AS finturk_81_il_farki_yuzde,
           round(tbb_minus_bddk_pct, 2) AS tbb_kapsam_farki_yuzde,
           disbursement_amount_million_try AS tbb_kullandirim_milyon_tl
    FROM analysis.housing_credit_quarterly
    ORDER BY quarter DESC
    LIMIT 8
    """
).fetchdf()


ceyrek,bddk_milyon_tl,finturk_81_il_farki_yuzde,tbb_kapsam_farki_yuzde,tbb_kullandirim_milyon_tl
2026-06,801438,0.0003,NaN,NaN
2026-03,741972,0.0005,-9.00,99520.736272
2025-12,678970,0.0007,-9.53,86982.706528
2025-09,623765,0.0008,-9.76,68306.996509
2025-06,583265,0.0009,-9.92,58246.678966
2025-03,548385,0.0010,-9.62,55340.637464
2024-12,512259,0.0011,-9.24,67391.647450
2024-09,465313,0.0016,-9.53,36915.694950


## İl bazlı konut panelinden örnek

In [ ]:
connection.execute(
    """
    SELECT province_name AS il,
           quarter AS ceyrek,
           housing_sales_total_count AS toplam_konut_satisi,
           round(mortgaged_sales_share_pct, 2) AS ipotekli_satis_payi_yuzde,
           round(housing_credit_per_capita_try, 2) AS kisi_basi_konut_kredisi_tl,
           round(housing_credit_yoy_pct, 2) AS konut_kredisi_yillik_yuzde,
           analysis_ready AS analize_hazir
    FROM regional.housing_quarterly
    WHERE quarter = '2026Q2'
      AND province_name IN ('ANKARA', 'İSTANBUL', 'İZMİR')
    ORDER BY province_name
    """
).fetchdf()


il,ceyrek,toplam_konut_satisi,ipotekli_satis_payi_yuzde,kisi_basi_konut_kredisi_tl,konut_kredisi_yillik_yuzde,analize_hazir
ANKARA,2026Q2,31169.0,25.50,18430.07,37.58,True
İSTANBUL,2026Q2,66133.0,23.18,14496.81,31.58,True
İZMİR,2026Q2,20704.0,22.02,13314.29,44.05,True


## Açık kalite notları

1. TBB Haziran 2026 tüketici kredileri raporu 8 Eylül 2026 kontrolünde kaynakta yayımlanmamıştır.
2. EVDS `TP.MK.KUL.YTL` serisinin Haziran 2026 gözlemi kaynak cevabında yoktur.
3. Eski EVDS `TP.ALTINPIYASA.KAP05` serisi seyrektir ve 24 Kasım 2025'te biter.
4. Aktif altın kontrolü `TP.ALTINPIYASA.KAP02` 30 Haziran 2026'ya kadar doludur; TL/gram serisi açık `0.001` dönüşümüyle türetilir.
5. Ağustos 2025 EVDS ve BDDK konut kredisi kapsam farkı otomatik düzeltilmez.
6. TBB raporlayan banka kapsamı BDDK sektör toplamından daha dardır.
7. Beş ilin konut birim fiyatı serisi tamamen boştur. Altı ilin serisi 2023'te,
   Şırnak serisi 2022'de başlar. Bu nedenle toplam 162 il-çeyrek birim fiyatı
   kaynakta yayımlanmadığı için null kalır.
8. Ham EVDS'deki 9 yarışma dönemi ipotekli satış null değeri, TÜİK'teki
   `toplam satış = diğer satış` özdeşliğiyle sıfır olarak doğrulanıp 8
   il-çeyrek toplamında açık fallback provenance ile kullanılır.

Bu boşluklar veri kaybı gibi gizlenmez. Katalog ve kalite tablolarında açıkça
görülebilir.


In [ ]:
connection.execute(
    """
    SELECT source_system AS kaynak,
           count(*) AS eksik_gozlemi_olan_metrik,
           sum(missing_observation_count) AS kaynak_bos_gozlem
    FROM catalog.metrics
    WHERE observation_available
      AND missing_observation_count > 0
    GROUP BY source_system
    ORDER BY source_system
    """
).fetchdf()


kaynak,eksik_gozlemi_olan_metrik,kaynak_bos_gozlem
BDDK_FINTURK,19,30892.0
REGIONAL_HOUSING_ANALYSIS,5,1600.0
TCMB_EVDS,27,2870.0
TCMB_EVDS_DERIVED,1,83.0
TUIK_DATA_PORTAL,1,10.0


In [ ]:
connection.close()
print('Bağlantı kapatıldı.')

## Sonuç

Veri toplama ve doğrulama aşaması, kaynakta yayımlanmış zorunlu kapsam için
tamamlandı. Bir sonraki ürün aşaması bu katalog ve DuckDB üzerinde generic
ingestion, güvenli sorgulama, analiz durumu, nedensellik iş akışları ve CloudX
agent katmanını kurmaktır.
